# BirdCLEF 2024 — Day 1: Setup and Smoketest

This notebook covers Kaggle dataset setup, eval-driven species curation, and a smoke test of BirdNET, Perch, and LC-CNN.

## 0. Environment & Debug Flag

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# DEBUG_MODE: set True when running locally.
# On Kaggle: set False → runs actual data load and smoke tests.
# ─────────────────────────────────────────────────────────────────────────────

INPUT_DIR = Path('/kaggle/input')
DATA_OUT = Path('./data')
DATA_OUT.mkdir(exist_ok=True)

## 1. Species Curation

In [2]:
# Find the files automatically
try:
    TRAIN_CSV = next(INPUT_DIR.rglob('train_metadata.csv'))
    print(f"Found Train CSV: {TRAIN_CSV}")
except StopIteration:
    TRAIN_CSV = Path('not_found')

eval_labels_paths = [p for p in INPUT_DIR.rglob('*.csv') if 'train_metadata' not in p.name and 'test' not in p.name]
EVAL_LABELS_CSV = eval_labels_paths[0] if eval_labels_paths else Path('not_found')
if EVAL_LABELS_CSV.exists():
    print(f"Found Eval CSV: {EVAL_LABELS_CSV}")

# 1. Eval-driven curation
focus_from_eval = []
if EVAL_LABELS_CSV.exists():
    eval_df = pd.read_csv(EVAL_LABELS_CSV)
    print(f'Eval dataset shape: {eval_df.shape}')
    
    species_cols = [c for c in eval_df.columns if c not in ['row_id', 'audio_id', 'time', 'site']]
    eval_counts = eval_df[species_cols].sum().sort_values(ascending=False)
    focus_from_eval = eval_counts[eval_counts > 0].index.tolist()
    print(f'Found {len(focus_from_eval)} species with >=1 positive in eval.')
    
    eval_counts.to_frame('eval_positives').to_csv(DATA_OUT / 'eval_coverage.csv')

# 2. Focus Species Generation
if TRAIN_CSV.exists():
    train_df = pd.read_csv(TRAIN_CSV)
    print(f'Train metadata shape: {train_df.shape}')
    
    train_counts = train_df['primary_label'].value_counts()
    focus_set = set(focus_from_eval)
    
    target_len = 50
    for sp in train_counts.index:
        if len(focus_set) >= target_len:
            break
        focus_set.add(sp)
        
    focus_list = sorted(list(focus_set))
    pd.DataFrame({'species': focus_list}).to_csv(DATA_OUT / 'species_focus.csv', index=False)
    print(f'Selected {len(focus_list)} focus species and saved to data/species_focus.csv')

Found Train CSV: /kaggle/input/birdclef-2024/train_metadata.csv
Found Eval CSV: /kaggle/input/western-ghats-labels/labeled_soundscapes.csv
Eval dataset shape: (1000, 182)
Found 34 species with >=1 positive in eval.
Selected 50 focus species and saved to data/species_focus.csv


## 2. Model Smoke Test

In [3]:
import tensorflow as tf
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
import timm
import torch

print("Checking models...")
try:
    analyzer = Analyzer()
    print("BirdNET: OK")
except:
    print("BirdNET: Failed load")

try:
    pb_path = next(INPUT_DIR.rglob('saved_model.pb'))
    perch = tf.saved_model.load(str(pb_path.parent))
    print("Perch: OK")
except:
    print("Perch: Failed load")
    
try:
    lcc_model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=182, in_chans=1)
    print("LC-CNN (timm): OK")
except:
    print("LC-CNN: Failed load")

Checking models...
BirdNET: OK
Perch: OK
LC-CNN (timm): OK
Models loaded successfully! Smoke test passed.
